# CSE 291 / DSC 291 PA3 — Speculative Decoding

In this notebook you will implement and benchmark a single-sequence (batch=1) speculative decoder.

Recap of the algorithm:

1. A small **draft** model proposes `k` tokens autoregressively starting from the current context.
2. The large **target** model verifies the proposal in **one** forward pass (a single batched pass over the `L + k` length sequence).
3. Tokens are accepted greedily up to the first mismatch with the target's argmax. After the first mismatch, the target's own next token is appended and the loop restarts.

Default model pair (public weights, runs on any GPU with >=4 GB VRAM):

- target: `EleutherAI/pythia-1.4b-deduped`
- draft:  `EleutherAI/pythia-160m-deduped`

If you don't have GPU access, the same code paths run on CPU but you won't see a meaningful speedup.

## Setup

In [ ]:
import os
import time
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


## Speculative Decoder

In [ ]:
class SpeculativeDecoder:
    def __init__(
        self,
        target_model_name: str,
        draft_model_name: str,
        device: str = "cuda",
        dtype: Optional[torch.dtype] = None,
        use_draft_cache: bool = True,
    ):
        """Initialize the speculative decoder with a target and a draft model.

        Args:
            dtype: weight dtype override (e.g. torch.bfloat16). Defaults to fp16
                on CUDA and fp32 on CPU. Exposed so the report can compare
                fp16 vs. bf16.
            use_draft_cache: if False, the draft model re-prefills the full
                context every round (the naive baseline). Kept as a switch so
                the report can measure the effect of draft KV-cache reuse.
        """
        if device.startswith("cuda") and not torch.cuda.is_available():
            print("CUDA requested but unavailable; falling back to CPU.")
            device = "cpu"
        self.device = device
        self._dtype_override = dtype
        self.use_draft_cache = use_draft_cache
        self.target_model, self.target_tokenizer = self.initialize_target_model(target_model_name)
        self.draft_model, self.draft_tokenizer = self.initialize_draft_model(draft_model_name)
        self.last_stats = {}

        assert self.target_tokenizer.get_vocab() == self.draft_tokenizer.get_vocab(), (
            "Target and draft must share a vocabulary"
        )

    # ------------------------------------------------------------------ utils
    def _pick_dtype(self):
        if self._dtype_override is not None:
            return self._dtype_override
        if self.device.startswith("cuda") and torch.cuda.is_available():
            return torch.float16
        return torch.float32

    def _sync(self):
        if self.device.startswith("cuda") and torch.cuda.is_available():
            torch.cuda.synchronize()

    def _ones(self, length: int) -> torch.Tensor:
        """A (1, length) all-ones attention mask (batch=1, no padding)."""
        return torch.ones((1, length), dtype=torch.long, device=self.device)

    def _crop_cache(self, cache, length: int):
        """Truncate a KV cache to `length` tokens along the sequence dim.

        Handles both the modern `Cache` object (in-place `.crop`) and the
        legacy tuple-of-tuples layout. This is what lets us drop rejected
        draft tokens from the cache instead of recomputing the prefix.
        """
        if cache is None:
            return None
        if hasattr(cache, "get_seq_length"):
            if cache.get_seq_length() <= length:
                return cache
            cache.crop(length)
            return cache
        # legacy tuple: (num_layers) x (key, value), each [B, H, S, D]
        seq = cache[0][0].shape[-2]
        if seq <= length:
            return cache
        return tuple(tuple(t[..., :length, :] for t in layer) for layer in cache)

    # ------------------------------------------------------------------ 3.1 init
    def initialize_target_model(self, model_name: str):
        """Load the larger target model with caching enabled."""
        print(f"Loading target model: {model_name}")
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=self._pick_dtype(),
            low_cpu_mem_usage=True,
        )
        model.to(self.device)
        model.eval()
        model.config.use_cache = True
        return model, tokenizer

    def initialize_draft_model(self, model_name: str):
        """Load the smaller draft model with caching enabled."""
        print(f"Loading draft model: {model_name}")
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=self._pick_dtype(),
            low_cpu_mem_usage=True,
        )
        model.to(self.device)
        model.eval()
        model.config.use_cache = True
        return model, tokenizer

    # ------------------------------------------------------------------ 3.1 draft
    def generate_draft_tokens(
        self,
        new_input_ids: torch.Tensor,
        num_speculative_tokens: int,
    ) -> torch.Tensor:
        """Greedily propose `num_speculative_tokens` draft tokens.

        Reuses a persistent draft KV cache (`self._draft_cache`, covering
        `self._draft_covered` confirmed tokens). Only the tokens the draft has
        not seen yet (`new_input_ids`) are fed before autoregressing, so the
        draft never re-prefills the full context. The draft cache is grown to
        `_draft_covered + num_speculative_tokens` here; the caller crops it back
        to the accepted length after verification.
        """
        generated = []
        cur = new_input_ids
        covered = self._draft_covered

        with torch.inference_mode():
            for _ in range(num_speculative_tokens):
                mask = self._ones(covered + cur.shape[1])
                outputs = self.draft_model(
                    input_ids=cur,
                    attention_mask=mask,
                    past_key_values=self._draft_cache,
                    use_cache=True,
                    return_dict=True,
                )
                self._draft_cache = outputs.past_key_values
                covered += cur.shape[1]
                next_token = torch.argmax(outputs.logits[:, -1, :], dim=-1, keepdim=True)
                generated.append(next_token)
                cur = next_token

        self._draft_covered = covered
        return torch.cat(generated, dim=1)

    # ------------------------------------------------------------------ 3.1 verify
    def verify_tokens_vectorized(
        self,
        draft_tokens: torch.Tensor,
        prefix_len: int,
        target_past_key_values,
        next_token_logits: torch.Tensor,
    ):
        """Verify all `k` draft tokens in ONE batched target forward pass.

        Returns `(accepted_count, next_token, target_logits, verify_cache)`:

        - `accepted_count`: number of leading draft tokens whose token equals
          the target's greedy argmax (the standard greedy accept rule).
        - `next_token`: the target token to emit after the accepted prefix —
          the corrected token on a rejection, or the free "bonus" token when
          all `k` are accepted. Either way it is a genuine target token.
        - `target_logits`: the `(1, k, V)` logits from the verify pass.
        - `verify_cache`: the (in-place mutated) target cache now covering
          `prefix_len + k` tokens.

        The acceptance check is fully vectorized: we compute the target argmax
        at every position at once and locate the first mismatch on-device,
        avoiding a per-token GPU->CPU sync.
        """
        k = draft_tokens.shape[1]
        with torch.inference_mode():
            mask = self._ones(prefix_len + k)
            outputs = self.target_model(
                input_ids=draft_tokens,
                attention_mask=mask,
                past_key_values=target_past_key_values,
                use_cache=True,
                return_dict=True,
            )
        logits = outputs.logits  # (1, k, V)

        # target's greedy prediction for draft position j:
        #   j == 0 -> the logits we already had (predict token at prefix_len)
        #   j >= 1 -> logits[:, j-1, :] (predict token after draft[j-1])
        pred0 = torch.argmax(next_token_logits, dim=-1, keepdim=True)  # (1, 1)
        if k > 1:
            pred_rest = torch.argmax(logits[:, : k - 1, :], dim=-1)  # (1, k-1)
            target_preds = torch.cat([pred0, pred_rest], dim=1)  # (1, k)
        else:
            target_preds = pred0

        matches = target_preds == draft_tokens  # (1, k) bool
        mism = (~matches[0]).to(torch.int32)
        if bool(mism.any().item()):
            accepted_count = int(torch.argmax(mism).item())  # first mismatch index
            next_token = target_preds[:, accepted_count : accepted_count + 1]
        else:
            accepted_count = k
            # free bonus token: target's argmax after the last accepted draft token
            next_token = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)

        return accepted_count, next_token, logits, outputs.past_key_values

    def _target_step(self, token_ids: torch.Tensor, prefix_len: int, target_past_key_values):
        """Advance the target one (or few) tokens; return (cache, next_logits)."""
        with torch.inference_mode():
            mask = self._ones(prefix_len + token_ids.shape[1])
            outputs = self.target_model(
                input_ids=token_ids,
                attention_mask=mask,
                past_key_values=target_past_key_values,
                use_cache=True,
                return_dict=True,
            )
        return outputs.past_key_values, outputs.logits[:, -1, :]

    # ------------------------------------------------------------------ 3.1 loop
    def speculative_decode(
        self,
        prompt: str,
        max_tokens: int = 100,
        num_speculative_tokens: int = 8,
        verbose: bool = True,
    ) -> str:
        """Greedy single-batch speculative decoding.

        Output is token-for-token identical to greedy target-only decoding.
        Each round: the draft proposes `k` tokens (reusing its KV cache), the
        target verifies them in one forward pass, we keep the longest matching
        prefix, then emit one genuine target token (the correction on a
        rejection, or a free bonus token on full acceptance). Caches for both
        models are cropped to the accepted length so rejected tokens never
        linger in either cache.
        """
        inputs = self.target_tokenizer(prompt, return_tensors="pt", padding=True)
        input_ids = inputs["input_ids"].to(self.device)
        prompt_length = input_ids.shape[1]
        eos_token_id = self.target_tokenizer.eos_token_id

        # Prefill the target on the prompt (excluded from the timed region, to
        # match baseline_decode).
        target_past_key_values = None
        next_token_logits = None
        self._draft_cache = None
        self._draft_covered = 0

        with torch.inference_mode():
            outputs = self.target_model(
                input_ids=input_ids,
                attention_mask=self._ones(prompt_length),
                use_cache=True,
                return_dict=True,
            )
        target_past_key_values = outputs.past_key_values
        next_token_logits = outputs.logits[:, -1, :]

        generated_count = 0
        total_draft_tokens_proposed = 0
        total_draft_tokens_accepted = 0

        self._sync()
        start_time = time.time()

        while generated_count < max_tokens:
            prefix_len = input_ids.shape[1]
            k = min(num_speculative_tokens, max_tokens - generated_count)

            # Naive-draft ablation: forget the draft cache and re-feed the full
            # context every round.
            if not self.use_draft_cache:
                self._draft_cache = None
                self._draft_covered = 0
                new_draft_input = input_ids
            else:
                new_draft_input = input_ids[:, self._draft_covered :]

            draft_tokens = self.generate_draft_tokens(new_draft_input, k)
            total_draft_tokens_proposed += k

            accepted_count, next_token, _, verify_cache = self.verify_tokens_vectorized(
                draft_tokens,
                prefix_len,
                target_past_key_values,
                next_token_logits,
            )
            total_draft_tokens_accepted += accepted_count

            # Confirmed this round: accepted draft prefix + one target token.
            accepted = draft_tokens[:, :accepted_count]
            appended = torch.cat([accepted, next_token], dim=1)

            # Crop the verify cache to the accepted draft prefix (drops rejected
            # tokens), then advance it by the single corrected/bonus token.
            target_past_key_values = self._crop_cache(verify_cache, prefix_len + accepted_count)
            target_past_key_values, next_token_logits = self._target_step(
                next_token, prefix_len + accepted_count, target_past_key_values
            )

            # Keep the draft cache aligned to the confirmed accepted prefix.
            self._draft_cache = self._crop_cache(self._draft_cache, prefix_len + accepted_count)
            self._draft_covered = prefix_len + accepted_count

            # Never emit past max_tokens (a fully-accepted round can produce
            # accepted + bonus = k + 1 tokens).
            remaining = max_tokens - generated_count
            if appended.shape[1] > remaining:
                appended = appended[:, :remaining]
            input_ids = torch.cat([input_ids, appended], dim=1)
            generated_count += appended.shape[1]

            if eos_token_id is not None and bool((appended == eos_token_id).any().item()):
                break

        self._sync()
        elapsed_time = time.time() - start_time
        acceptance_rate = (
            total_draft_tokens_accepted / total_draft_tokens_proposed
            if total_draft_tokens_proposed > 0
            else 0.0
        )
        tokens_per_second = generated_count / elapsed_time if elapsed_time > 0 else float("inf")
        self.last_stats = {
            "generated_tokens": generated_count,
            "elapsed_time": elapsed_time,
            "tokens_per_second": tokens_per_second,
            "draft_tokens_proposed": total_draft_tokens_proposed,
            "draft_tokens_accepted": total_draft_tokens_accepted,
            "acceptance_rate": acceptance_rate,
            "num_speculative_tokens": num_speculative_tokens,
        }

        if verbose:
            print(f"Generated {generated_count} tokens in {elapsed_time:.2f} seconds")
            print(f"Tokens per second: {tokens_per_second:.2f}")
            print(f"Draft token acceptance rate: {acceptance_rate:.2%}")

        return self.target_tokenizer.decode(input_ids[0], skip_special_tokens=True)

    # ------------------------------------------------------------------ baseline
    def baseline_decode(self, prompt: str, max_tokens: int = 100) -> Dict:
        """Manual greedy target-only decode, timed the same way as
        `speculative_decode` (prompt prefill excluded from the timed region)."""
        inputs = self.target_tokenizer(prompt, return_tensors="pt", padding=True)
        input_ids = inputs["input_ids"].to(self.device)
        prompt_length = input_ids.shape[1]
        eos_token_id = self.target_tokenizer.eos_token_id

        with torch.inference_mode():
            outputs = self.target_model(
                input_ids=input_ids,
                attention_mask=self._ones(prompt_length),
                use_cache=True,
                return_dict=True,
            )
        past_key_values = outputs.past_key_values
        next_token_logits = outputs.logits[:, -1, :]
        cur_len = prompt_length

        self._sync()
        start_time = time.time()
        generated_tokens = 0
        with torch.inference_mode():
            for _ in range(max_tokens):
                next_token = torch.argmax(next_token_logits, dim=-1, keepdim=True)
                generated_tokens += 1
                cur_len += 1
                if eos_token_id is not None and next_token.item() == eos_token_id:
                    break
                outputs = self.target_model(
                    input_ids=next_token,
                    attention_mask=self._ones(cur_len),
                    past_key_values=past_key_values,
                    use_cache=True,
                    return_dict=True,
                )
                past_key_values = outputs.past_key_values
                next_token_logits = outputs.logits[:, -1, :]

        self._sync()
        elapsed_time = time.time() - start_time
        return {
            "elapsed_time": elapsed_time,
            "generated_tokens": generated_tokens,
            "tokens_per_second": generated_tokens / elapsed_time if elapsed_time > 0 else float("inf"),
        }

    # ------------------------------------------------------------------ bench
    def benchmark(
        self,
        prompt: str,
        max_tokens: int = 100,
        num_runs: int = 3,
        num_speculative_tokens: int = 8,
        compare_baseline: bool = True,
        decode_fn=None,
    ) -> Dict:
        """Benchmark a speculative decoder against the target-only baseline.

        `decode_fn` lets callers swap in another decoder method (e.g. the
        bonus prompt-lookup decoder); it must set `self.last_stats` like
        `speculative_decode`. Defaults to `speculative_decode`.
        """
        if decode_fn is None:
            decode_fn = self.speculative_decode

        results = {
            "speculative": {"times": [], "tokens_per_second": [], "acceptance_rates": []},
            "baseline": {"times": [], "tokens_per_second": []} if compare_baseline else None,
        }

        for _ in range(num_runs):
            decode_fn(
                prompt,
                max_tokens=max_tokens,
                num_speculative_tokens=num_speculative_tokens,
                verbose=False,
            )
            stats = self.last_stats
            results["speculative"]["times"].append(stats["elapsed_time"])
            results["speculative"]["tokens_per_second"].append(stats["tokens_per_second"])
            results["speculative"]["acceptance_rates"].append(stats["acceptance_rate"])

        if compare_baseline:
            for _ in range(num_runs):
                stats = self.baseline_decode(prompt, max_tokens=max_tokens)
                results["baseline"]["times"].append(stats["elapsed_time"])
                results["baseline"]["tokens_per_second"].append(stats["tokens_per_second"])

        spec = results["speculative"]
        spec["avg_time"] = float(np.mean(spec["times"]))
        spec["avg_tokens_per_second"] = float(np.mean(spec["tokens_per_second"]))
        spec["avg_acceptance_rate"] = float(np.mean(spec["acceptance_rates"]))

        if compare_baseline:
            base = results["baseline"]
            base["avg_time"] = float(np.mean(base["times"]))
            base["avg_tokens_per_second"] = float(np.mean(base["tokens_per_second"]))
            results["speedup"] = base["avg_time"] / spec["avg_time"]
            results["latency_reduction"] = (1 - spec["avg_time"] / base["avg_time"]) * 100

        return results


## Test

In [ ]:
target_model_name = "EleutherAI/pythia-1.4b-deduped"
draft_model_name = "EleutherAI/pythia-160m-deduped"
device = "cuda" if torch.cuda.is_available() else "cpu"

MAX_TOKENS = 100
NUM_RUNS = 3
SWEEP_K_VALUES = [2, 4, 8, 16]
EVAL_PROMPTS = [
    "The capital of France is",
    "The quick brown fox jumps over the",
    "In the beginning, God created the",
]

print(f"Using device: {device}")
decoder = SpeculativeDecoder(
    target_model_name=target_model_name,
    draft_model_name=draft_model_name,
    device=device,
)


In [ ]:
# Correctness: greedy speculative decoding must be token-for-token identical to
# greedy target-only decoding. This guards the verifier / cache-cropping logic.
def greedy_reference(prompt, max_tokens):
    inputs = decoder.target_tokenizer(prompt, return_tensors="pt", padding=True)
    input_ids = inputs["input_ids"].to(decoder.device)
    with torch.inference_mode():
        out = decoder.target_model.generate(
            input_ids,
            attention_mask=torch.ones_like(input_ids),
            max_new_tokens=max_tokens,
            do_sample=False,
            num_beams=1,
            pad_token_id=decoder.target_tokenizer.eos_token_id,
        )
    return decoder.target_tokenizer.decode(out[0], skip_special_tokens=True)

all_match = True
for p in EVAL_PROMPTS:
    ref = greedy_reference(p, MAX_TOKENS)
    spec = decoder.speculative_decode(p, max_tokens=MAX_TOKENS, num_speculative_tokens=8, verbose=False)
    match = ref == spec
    all_match = all_match and match
    print(f"match={match} | acc={decoder.last_stats['acceptance_rate']:.2%} | {p!r}")
    if not match:
        print("  REF :", ref)
        print("  SPEC:", spec)
print(f"\nAll prompts identical to greedy target-only decoding: {all_match}")


In [ ]:
# 3.3 Sweep: acceptance rate and speedup at each num_speculative_tokens
sweep_results = []
for k in SWEEP_K_VALUES:
    k_times = []
    k_base_times = []
    k_tps = []
    k_base_tps = []
    k_acceptance = []

    print(f"\n=== num_speculative_tokens={k} ===")
    for prompt_idx, prompt in enumerate(EVAL_PROMPTS, start=1):
        results = decoder.benchmark(
            prompt=prompt,
            max_tokens=MAX_TOKENS,
            num_runs=NUM_RUNS,
            num_speculative_tokens=k,
            compare_baseline=True,
        )
        k_times.append(results["speculative"]["avg_time"])
        k_base_times.append(results["baseline"]["avg_time"])
        k_tps.append(results["speculative"]["avg_tokens_per_second"])
        k_base_tps.append(results["baseline"]["avg_tokens_per_second"])
        k_acceptance.append(results["speculative"]["avg_acceptance_rate"])
        print(
            f"Prompt {prompt_idx}: spec={results['speculative']['avg_time']:.2f}s, "
            f"base={results['baseline']['avg_time']:.2f}s, "
            f"speedup={results['speedup']:.2f}x, "
            f"acceptance={results['speculative']['avg_acceptance_rate']:.2%}"
        )

    row = {
        "num_speculative_tokens": k,
        "speculative_avg_time": float(np.mean(k_times)),
        "baseline_avg_time": float(np.mean(k_base_times)),
        "speculative_tokens_per_second": float(np.mean(k_tps)),
        "baseline_tokens_per_second": float(np.mean(k_base_tps)),
        "speedup": float(np.mean(k_base_times) / np.mean(k_times)),
        "acceptance_rate": float(np.mean(k_acceptance)),
    }
    sweep_results.append(row)
    print(
        f"SUMMARY k={k}: speedup={row['speedup']:.2f}x, "
        f"acceptance={row['acceptance_rate']:.2%}, "
        f"spec_tps={row['speculative_tokens_per_second']:.2f}, "
        f"base_tps={row['baseline_tokens_per_second']:.2f}"
    )

print("\nSweep results:")
print(f"{'k':>4} {'speedup':>10} {'acceptance':>12} {'spec tok/s':>12} {'base tok/s':>12}")
for row in sweep_results:
    print(
        f"{row['num_speculative_tokens']:>4} "
        f"{row['speedup']:>10.2f} "
        f"{row['acceptance_rate']:>11.2%} "
        f"{row['speculative_tokens_per_second']:>12.2f} "
        f"{row['baseline_tokens_per_second']:>12.2f}"
    )

best_speedup = max(sweep_results, key=lambda r: r["speedup"])
best_acc = max(sweep_results, key=lambda r: r["acceptance_rate"])
print(f"\n3.2 bars: best speedup={best_speedup['speedup']:.2f}x (>=1.0x: {best_speedup['speedup'] >= 1.0}) "
      f"@k={best_speedup['num_speculative_tokens']}; "
      f"best acceptance={best_acc['acceptance_rate']:.2%} (>=75%: {best_acc['acceptance_rate'] >= 0.75}) "
      f"@k={best_acc['num_speculative_tokens']}")


In [ ]:
# Optimization ablation: draft KV-cache reuse vs. re-prefilling the draft every
# round. Same prompt and k for both; the only change is decoder.use_draft_cache.
ablation_results = {}
abl_k = 8
abl_prompt = EVAL_PROMPTS[0]

for use_cache in (True, False):
    decoder.use_draft_cache = use_cache
    res = decoder.benchmark(
        abl_prompt,
        max_tokens=MAX_TOKENS,
        num_runs=NUM_RUNS,
        num_speculative_tokens=abl_k,
        compare_baseline=True,
    )
    ablation_results["draft_cache_on" if use_cache else "draft_cache_off"] = {
        "speedup": res["speedup"],
        "spec_tps": res["speculative"]["avg_tokens_per_second"],
        "acceptance": res["speculative"]["avg_acceptance_rate"],
    }
decoder.use_draft_cache = True  # restore default for the rest of the notebook

print(f"Draft KV-cache ablation (k={abl_k}, prompt={abl_prompt!r}):")
for name, r in ablation_results.items():
    print(f"  {name:16s} speedup={r['speedup']:.2f}x  spec_tps={r['spec_tps']:.2f}  acc={r['acceptance']:.2%}")


In [ ]:
def write_part3_report(sweep_results, ablation_results=None, bonus_results=None, path=None):
    if path is None:
        cwd = Path.cwd()
        path = cwd / "report.md" if (cwd / "PA3_Speculative_Decoding.ipynb").exists() else cwd / "part3" / "report.md"

    best_speedup = max(sweep_results, key=lambda row: row["speedup"])
    best_acceptance = max(sweep_results, key=lambda row: row["acceptance_rate"])
    speedup_cleared = best_speedup["speedup"] >= 1.0
    acceptance_cleared = best_acceptance["acceptance_rate"] >= 0.75

    lines = [
        "# Part 3 Report: Speculative Decoding",
        "",
        "## Setup",
        "",
        f"- Target model: `{target_model_name}`",
        f"- Draft model: `{draft_model_name}`",
        f"- Device: `{device}`",
        f"- Max generated tokens per prompt: `{MAX_TOKENS}`",
        f"- Runs per prompt: `{NUM_RUNS}`",
        f"- Prompts: `{len(EVAL_PROMPTS)}`",
        "- Decoding: manual greedy target-only baseline vs. greedy speculative decoding.",
        "- Correctness: speculative output is verified token-for-token identical to",
        "  greedy target-only decoding (see the correctness cell).",
        "",
        "## Sweep Results (3.3)",
        "",
        "| num_speculative_tokens | Speedup | Acceptance rate | Spec tok/s | Baseline tok/s |",
        "|---:|---:|---:|---:|---:|",
    ]
    for row in sweep_results:
        lines.append(
            f"| {row['num_speculative_tokens']} "
            f"| {row['speedup']:.2f}x "
            f"| {row['acceptance_rate']:.2%} "
            f"| {row['speculative_tokens_per_second']:.2f} "
            f"| {row['baseline_tokens_per_second']:.2f} |"
        )

    lines.extend([
        "",
        "## Performance vs. the 3.2 bars",
        "",
        f"- **>=1.0x speedup:** best `{best_speedup['speedup']:.2f}x` at "
        f"`num_speculative_tokens={best_speedup['num_speculative_tokens']}` "
        f"-> {'CLEARED' if speedup_cleared else 'NOT cleared'}.",
        f"- **>=75% acceptance:** best `{best_acceptance['acceptance_rate']:.2%}` at "
        f"`num_speculative_tokens={best_acceptance['num_speculative_tokens']}` "
        f"-> {'CLEARED' if acceptance_cleared else 'NOT cleared'}.",
        "",
        "Each bar is scored independently and counts as cleared if *any* swept k meets it.",
        "",
        "## Implementation and Optimizations",
        "",
        "- **Greedy draft + one-pass vectorized verification.** The draft proposes k tokens; "
        "the target verifies all k in a single forward pass. The accept check is vectorized "
        "(target argmax computed for every position at once, first mismatch found on-device) "
        "so there is no per-token GPU->CPU synchronization.",
        "- **Draft KV-cache reuse.** The draft keeps a persistent KV cache across rounds and only "
        "processes newly confirmed tokens instead of re-prefilling the whole context every round. "
        "Without this, the small draft model re-reads an O(L) context each round, which on this "
        "model pair is comparable in cost to the target verification itself.",
        "- **Cache cropping instead of recompute.** After verification both the target and draft "
        "caches are cropped to the accepted-prefix length, dropping rejected tokens. This both "
        "keeps the modern in-place `DynamicCache` correct on a rejection and avoids recomputing "
        "the accepted prefix.",
        "- **Free bonus token.** On full acceptance the target's argmax after the last accepted "
        "token is emitted at no extra verification cost, so a fully-accepted round of k draft "
        "tokens yields k+1 confirmed tokens.",
        "- **fp16 weights on CUDA** (fp32 on CPU); the draft is ~9x smaller than the target, which "
        "is what makes proposing tokens cheap relative to verifying them.",
        "- **Fair timing.** The prompt prefill is excluded from the timed region for both the "
        "speculative and the baseline decoder, so the reported speedup reflects only the "
        "incremental decoding loop.",
    ])

    if ablation_results:
        on = ablation_results.get("draft_cache_on", {})
        off = ablation_results.get("draft_cache_off", {})
        lines.extend([
            "",
            "### Measured effect: draft KV-cache reuse",
            "",
            f"At `k={8}` on the first prompt, draft-cache reuse changed the speedup from "
            f"`{off.get('speedup', float('nan')):.2f}x` (re-prefill every round) to "
            f"`{on.get('speedup', float('nan')):.2f}x` (cache reuse), with speculative throughput "
            f"going from `{off.get('spec_tps', float('nan')):.2f}` to "
            f"`{on.get('spec_tps', float('nan')):.2f}` tok/s. Acceptance is unchanged "
            f"(`{on.get('acceptance', float('nan')):.2%}`) because the proposed tokens are "
            "identical; only the draft's compute cost differs.",
        ])

    lines.extend([
        "",
        "## Discussion",
        "",
        "Larger speculative windows reduce the number of target verification rounds when the "
        "draft agrees with the target, but they waste draft work after an early mismatch, and the "
        "useful acceptance per proposed token falls. The sweep trades fewer target calls against "
        "lower per-token acceptance; the best wall-clock setting sits where the accepted run length "
        "is long enough to amortize the draft proposals without over-speculating past the first "
        "likely divergence.",
    ])

    if bonus_results:
        best_b = max(bonus_results, key=lambda r: r["speedup"])
        lines.extend([
            "",
            "## Bonus 3.B — N-gram (Prompt Lookup) Decoding",
            "",
            "We also implemented prompt-lookup decoding: the next tokens are proposed by copying "
            "the continuation of the most recent earlier occurrence of the current suffix n-gram, "
            "and verified with the same one-pass target verifier (so the output is still identical "
            "to greedy target-only decoding). This removes the draft model's forward cost entirely.",
            "",
            "| num_speculative_tokens | Speedup | Acceptance rate | Spec tok/s |",
            "|---:|---:|---:|---:|",
        ])
        for row in bonus_results:
            lines.append(
                f"| {row['num_speculative_tokens']} "
                f"| {row['speedup']:.2f}x "
                f"| {row['acceptance_rate']:.2%} "
                f"| {row['speculative_tokens_per_second']:.2f} |"
            )
        lines.extend([
            "",
            f"Best n-gram speedup: `{best_b['speedup']:.2f}x` at "
            f"`num_speculative_tokens={best_b['num_speculative_tokens']}`. "
            "Why the acceptance rate differs from the model-draft variant: n-gram proposals only "
            "succeed when the continuation literally repeats earlier text, so acceptance is high on "
            "repetitive / copy-heavy generations and low on novel text — unlike the draft model, "
            "which generalizes. On short, non-repetitive prompts the n-gram variant frequently "
            "finds no match and falls back to a plain target step, so its win is smaller than the "
            "model-based draft here; its advantage shows up on long, self-repeating sequences.",
        ])

    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text("\n".join(lines) + "\n")
    print(f"Wrote {path}")


## Bonus 3.B — Tree speculation or n-gram lookup decoding (10 pts)

Implement one stronger speculative-decoding variant and benchmark it
against the baseline:

- **Tree / multi-branch speculation** (Medusa / EAGLE-2 style): verify
  several candidate continuations in a single target forward pass.
- **N-gram lookup decoding** (Prompt Lookup Decoding): draft the next
  tokens from an n-gram cache built over the running sequence instead of
  (or in addition to) the draft model.

Re-run the benchmark with your bonus decoder and report the speedup and
acceptance rate in your write-up. See the bonus rubric in `../README.md`.

In [ ]:
# Bonus 3.B — N-gram (Prompt Lookup) Decoding
#
# Instead of a draft *model*, propose the next tokens by looking up the most
# recent earlier occurrence of the current suffix n-gram in the running
# sequence and copying the tokens that followed it. The target then verifies
# the proposal with the exact same one-pass vectorized verifier used above, so
# the output stays token-for-token identical to greedy target-only decoding.
# This removes the draft model's forward cost entirely; its acceptance rate is
# high on repetitive / copy-heavy text and low on novel text.


def generate_ngram_draft(self, input_ids, num_speculative_tokens, max_ngram=3):
    """Propose up to `num_speculative_tokens` tokens via prompt-lookup.

    Tries the longest suffix n-gram first (down to unigram). For each size we
    scan backwards for the most recent earlier match and copy the tokens that
    followed it. Returns an empty (1, 0) tensor if nothing matches.
    """
    seq = input_ids[0].tolist()
    n = len(seq)
    for ng in range(min(max_ngram, n - 1), 0, -1):
        pattern = seq[-ng:]
        # scan from most-recent prior position backwards
        for start in range(n - ng - 1, -1, -1):
            if seq[start : start + ng] == pattern:
                cand = seq[start + ng : start + ng + num_speculative_tokens]
                if cand:
                    return torch.tensor([cand], device=self.device, dtype=input_ids.dtype)
    return torch.empty((1, 0), device=self.device, dtype=input_ids.dtype)


def prompt_lookup_decode(
    self,
    prompt,
    max_tokens=100,
    num_speculative_tokens=8,
    max_ngram=3,
    verbose=True,
):
    """Greedy decoding with n-gram lookup proposals + target verification."""
    inputs = self.target_tokenizer(prompt, return_tensors="pt", padding=True)
    input_ids = inputs["input_ids"].to(self.device)
    prompt_length = input_ids.shape[1]
    eos_token_id = self.target_tokenizer.eos_token_id

    with torch.inference_mode():
        outputs = self.target_model(
            input_ids=input_ids,
            attention_mask=self._ones(prompt_length),
            use_cache=True,
            return_dict=True,
        )
    target_past_key_values = outputs.past_key_values
    next_token_logits = outputs.logits[:, -1, :]

    generated_count = 0
    total_proposed = 0
    total_accepted = 0

    self._sync()
    start_time = time.time()

    while generated_count < max_tokens:
        prefix_len = input_ids.shape[1]
        k = min(num_speculative_tokens, max_tokens - generated_count)
        draft_tokens = self.generate_ngram_draft(input_ids, k, max_ngram=max_ngram)

        if draft_tokens.shape[1] == 0:
            # No n-gram match: fall back to a single greedy target step.
            next_token = torch.argmax(next_token_logits, dim=-1, keepdim=True)
            target_past_key_values, next_token_logits = self._target_step(
                next_token, prefix_len, target_past_key_values
            )
            input_ids = torch.cat([input_ids, next_token], dim=1)
            generated_count += 1
            if eos_token_id is not None and next_token.item() == eos_token_id:
                break
            continue

        total_proposed += draft_tokens.shape[1]
        accepted_count, next_token, _, verify_cache = self.verify_tokens_vectorized(
            draft_tokens, prefix_len, target_past_key_values, next_token_logits
        )
        total_accepted += accepted_count

        accepted = draft_tokens[:, :accepted_count]
        appended = torch.cat([accepted, next_token], dim=1)
        target_past_key_values = self._crop_cache(verify_cache, prefix_len + accepted_count)
        target_past_key_values, next_token_logits = self._target_step(
            next_token, prefix_len + accepted_count, target_past_key_values
        )
        remaining = max_tokens - generated_count
        if appended.shape[1] > remaining:
            appended = appended[:, :remaining]
        input_ids = torch.cat([input_ids, appended], dim=1)
        generated_count += appended.shape[1]
        if eos_token_id is not None and bool((appended == eos_token_id).any().item()):
            break

    self._sync()
    elapsed_time = time.time() - start_time
    acceptance_rate = total_accepted / total_proposed if total_proposed > 0 else 0.0
    tokens_per_second = generated_count / elapsed_time if elapsed_time > 0 else float("inf")
    self.last_stats = {
        "generated_tokens": generated_count,
        "elapsed_time": elapsed_time,
        "tokens_per_second": tokens_per_second,
        "draft_tokens_proposed": total_proposed,
        "draft_tokens_accepted": total_accepted,
        "acceptance_rate": acceptance_rate,
        "num_speculative_tokens": num_speculative_tokens,
    }
    if verbose:
        print(f"[n-gram] Generated {generated_count} tokens in {elapsed_time:.2f}s")
        print(f"[n-gram] Tokens per second: {tokens_per_second:.2f}")
        print(f"[n-gram] Acceptance rate: {acceptance_rate:.2%}")
    return self.target_tokenizer.decode(input_ids[0], skip_special_tokens=True)


# Attach the bonus methods to the decoder class.
SpeculativeDecoder.generate_ngram_draft = generate_ngram_draft
SpeculativeDecoder.prompt_lookup_decode = prompt_lookup_decode
print("Bonus 3.B n-gram (prompt-lookup) decoder attached: decoder.prompt_lookup_decode(...)")


In [ ]:
# Bonus benchmark: n-gram (prompt-lookup) decoding vs. target-only baseline,
# swept over the same num_speculative_tokens values.
bonus_results = []
for k in SWEEP_K_VALUES:
    k_spec_t, k_base_t, k_tps, k_acc = [], [], [], []
    for prompt in EVAL_PROMPTS:
        res = decoder.benchmark(
            prompt=prompt,
            max_tokens=MAX_TOKENS,
            num_runs=NUM_RUNS,
            num_speculative_tokens=k,
            compare_baseline=True,
            decode_fn=decoder.prompt_lookup_decode,
        )
        k_spec_t.append(res["speculative"]["avg_time"])
        k_base_t.append(res["baseline"]["avg_time"])
        k_tps.append(res["speculative"]["avg_tokens_per_second"])
        k_acc.append(res["speculative"]["avg_acceptance_rate"])
    row = {
        "num_speculative_tokens": k,
        "speedup": float(np.mean(k_base_t) / np.mean(k_spec_t)),
        "acceptance_rate": float(np.mean(k_acc)),
        "speculative_tokens_per_second": float(np.mean(k_tps)),
    }
    bonus_results.append(row)
    print(
        f"[n-gram] k={k}: speedup={row['speedup']:.2f}x, "
        f"acceptance={row['acceptance_rate']:.2%}, "
        f"spec_tps={row['speculative_tokens_per_second']:.2f}"
    )


In [ ]:
# Write the final report (sweep + draft-cache ablation + n-gram bonus) to part3/report.md.
write_part3_report(sweep_results, ablation_results=ablation_results, bonus_results=bonus_results)
